# TAM Penetration Counts — Testing & Manual Insert

**Notebook** for inspecting and (optionally) writing to `cdt.tam_penetration_counts`.

## Layout

| Section | Cells | Default (Run All) | Purpose |
|---------|-------|-------------------|---------|
| **Setup** | 1–6 | ✅ Always runs | Imports, config, helpers, table check, pre-flight |
| **Write Guard** | 7 | ✅ Always runs | Sets `WRITE_MODE = False` (read-only by default) |
| **Write** | 8–9 | ⏭ Skipped | DELETE + INSERT (requires `WRITE_MODE = True`) |
| **Validation** | 10–13 | ✅ Always runs | 6×6 check, day-over-day, table summary, full data |

## How to use

- **Inspect only (default):** Run All → see existing data, validate, summarize. No writes happen.
- **Write + validate:** Set `WRITE_MODE = True` in cell 7, then Run All.
- **Override date:** Uncomment the override line in cell 3.

In [ ]:
import os
import json
import subprocess as sp
from io import StringIO
from datetime import date, timedelta
import pandas as pd
from jinja2 import Environment, FileSystemLoader
import pendulum

# Airflow Imports (commented out for notebook usage)
# from airflow import DAG
# from airflow.operators.python import PythonOperator
# from airflow.utils.email import send_email

print(f"Pandas: {pd.__version__}")

In [ ]:
# --- Configuration ---
DAG_HOME = os.path.dirname(os.path.abspath("__file__"))
TABLE_NAME = "tam_penetration_counts"
pacific_tz = pendulum.timezone("America/Los_Angeles")

email_list = [
    "huzefa.saifee@workday.com",
    "m6a0l2y5u3c9i6f3@workday.enterprise.slack.com",
]

# --- Effective Date Computation ---
# Same heuristic as the DAG's manual-trigger fallback:
#   Pacific hour < 22 → yesterday
#   Pacific hour >= 22 → today
now_pacific = pendulum.now(pacific_tz)
if now_pacific.hour < 22:
    effective_date = str(now_pacific.subtract(days=1).date())
else:
    effective_date = str(now_pacific.date())

# Uncomment to override with a specific date:
# effective_date = "2026-04-26"

print(f"Current Pacific time: {now_pacific.format('YYYY-MM-DD HH:mm:ss')}")
print(f"Effective snapshot date: {effective_date}")

In [ ]:
# --- Helpers (mirroring the DAG exactly) ---

jinja_env = Environment(loader=FileSystemLoader(DAG_HOME))


def render_sql(filename, **kwargs):
    """Loads and renders a Jinja2 template SQL file."""
    template = jinja_env.get_template(filename)
    return template.render(**kwargs)


def run_cli(cmd, fetch_data=False):
    """Executes a pharos CLI command."""
    result = sp.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed with code {result.returncode}\nCMD: {cmd}\nSTDERR: {result.stderr}"
        )

    raw_output = result.stdout.strip()
    if fetch_data:
        try:
            parsed = json.loads(raw_output)
            return parsed["result"]["data"]
        except json.JSONDecodeError as e:
            raise RuntimeError(
                f"Failed to parse JSON output. Command: {cmd}\nOutput: {raw_output[:300]}"
            ) from e
    return raw_output


def build_insert_query(current_date_str):
    """Assemble the full INSERT INTO ... WITH ... SELECT query."""
    cte_sql = render_sql("penetration_ctes.sql", current_date=current_date_str)
    select_sql = render_sql(
        "compute_select.sql",
        comments="Manual daily snapshot",
        current_date=current_date_str,
    )
    return f"INSERT INTO cdt.{TABLE_NAME}\nWITH\n{cte_sql}\n{select_sql}"


# Verify templates exist
for f in ['penetration_ctes.sql', 'compute_select.sql', 'create_table.sql']:
    assert os.path.exists(os.path.join(DAG_HOME, f)), f"Missing: {f}"
print("SQL templates OK")

In [ ]:
# --- Ensure Table Exists + Check Dependency ---
tables_csv = run_cli('pharos sql run --sql "SHOW TABLES IN dw.cdt"', fetch_data=True)
tables = tables_csv.split("\n")

if "workday_go_accounts" not in tables:
    raise RuntimeError(
        "Required dependency cdt.workday_go_accounts is missing. "
        "Run the workday_go_accounts notebook first."
    )
print("Dependency cdt.workday_go_accounts: OK")

create_ddl = render_sql("create_table.sql", table_name=TABLE_NAME)
if TABLE_NAME not in tables:
    print(f"Table cdt.{TABLE_NAME} not found. Creating...")
    run_cli(f'pharos sql run --sql "{create_ddl}"', fetch_data=False)
else:
    try:
        run_cli(
            f'pharos sql run --sql "SELECT COUNT(*) FROM cdt.{TABLE_NAME} WHERE snapshot_date IS NOT NULL LIMIT 1"',
            fetch_data=True,
        )
        print(f"Table cdt.{TABLE_NAME}: exists and is healthy.")
    except RuntimeError:
        print(f"Table cdt.{TABLE_NAME} is broken. Dropping and recreating...")
        run_cli(f'pharos sql run --sql "DROP TABLE IF EXISTS cdt.{TABLE_NAME}"', fetch_data=False)
        run_cli(f'pharos sql run --sql "{create_ddl}"', fetch_data=False)

In [ ]:
# --- Pre-flight: check if data already exists for effective_date (idempotent overwrite) ---
try:
    existing_csv = run_cli(
        f'pharos sql run --sql "SELECT market_segment, active_customer_count FROM cdt.{TABLE_NAME} WHERE snapshot_date = DATE \'{effective_date}\' ORDER BY market_segment"',
        fetch_data=True,
    )
    df_existing = pd.read_csv(StringIO(existing_csv))
    if len(df_existing) > 0:
        print(f"Found {len(df_existing)} rows for {effective_date}:")
        print(df_existing.to_string(index=False))
    else:
        print(f"No existing data for {effective_date}.")
except Exception:
    print(f"No existing data for {effective_date}.")

In [ ]:
# --- Write Mode (default: False — Run All is read-only) ---
WRITE_MODE = False

print(f"WRITE_MODE: {WRITE_MODE} {'— writes ENABLED' if WRITE_MODE else '— read-only (safe)'}")

---
## ⚠️ Write Section

The cell below performs **DELETE + INSERT** on `cdt.tam_penetration_counts`.

- **Skipped by default** when `WRITE_MODE = False` (set in cell right above).
- To enable: set `WRITE_MODE = True` in cell above, then re-run from there.

In [ ]:
# --- DELETE + INSERT (guarded by WRITE_MODE) ---
if WRITE_MODE:
    print(f"WRITE_MODE is ON — proceeding with DELETE + INSERT for {effective_date}")

    print(f"Deleting existing partition for {effective_date} (idempotent)...")
    run_cli(
        f'pharos sql run --sql "DELETE FROM cdt.{TABLE_NAME} WHERE snapshot_date = DATE \'{effective_date}\'"',
        fetch_data=False,
    )

    full_query = build_insert_query(effective_date)
    print(f"Executing penetration counts query for {effective_date}...")
    run_cli(f'pharos sql run --sql "{full_query}"', fetch_data=False)

    print(f"Snapshot for {effective_date} inserted successfully.")
else:
    print("WRITE_MODE is False — skipping DELETE + INSERT (read-only mode).")
    print("To enable writes, set WRITE_MODE = True in cell 7.")

---
## Validation

These cells run regardless of `WRITE_MODE`. They inspect whatever data currently
exists in the table for the effective date.

In [ ]:
# --- Validation 1: Verify 6 rows x 6 segments for effective_date ---
v_csv = run_cli(
    f'pharos sql run --sql "SELECT market_segment, active_customer_count, active_deployment_count_all, activity_initial_tool_usage, activity_initial_migrated, snapshot_date FROM cdt.{TABLE_NAME} WHERE snapshot_date = DATE \'{effective_date}\' ORDER BY market_segment"',
    fetch_data=True,
)
df_today = pd.read_csv(StringIO(v_csv))

expected_segments = {'All', 'GO Customers', 'GO Partners', 'LE', 'Launch/Express', 'ME'}
actual_segments = set(df_today['market_segment'].unique())

print(f"Rows for {effective_date}: {len(df_today)}")
if len(df_today) == 6 and actual_segments == expected_segments:
    print("CHECK 1 PASSED: 6 rows, all 6 segments present")
else:
    print(f"CHECK 1 FAILED: got {len(df_today)} rows, segments: {actual_segments}")

df_today

In [ ]:
# --- Validation 2: Day-over-day comparison ---
prev_date = str(date.fromisoformat(effective_date) - timedelta(days=1))

try:
    prev_csv = run_cli(
        f'pharos sql run --sql "SELECT market_segment, active_customer_count, active_deployment_count_all, activity_initial_tool_usage, activity_initial_migrated FROM cdt.{TABLE_NAME} WHERE snapshot_date = DATE \'{prev_date}\' ORDER BY market_segment"',
        fetch_data=True,
    )
    df_prev = pd.read_csv(StringIO(prev_csv))

    if df_prev.empty:
        print(f"No data for {prev_date} — skipping day-over-day comparison.")
    else:
        compare_cols = ['active_customer_count', 'active_deployment_count_all',
                        'activity_initial_tool_usage', 'activity_initial_migrated']

        merged = df_today.merge(
            df_prev, on='market_segment', suffixes=('_today', '_prev')
        )

        print(f"Day-over-day comparison ({prev_date} → {effective_date}):")
        print(f"{'Segment':<18} {'Metric':<30} {'Prev':>8} {'Today':>8} {'Delta':>8} {'%':>8}")
        print("-" * 90)

        anomalies = []
        for _, row in merged.iterrows():
            seg = row['market_segment']
            for col in compare_cols:
                prev_val = row[f'{col}_prev']
                today_val = row[f'{col}_today']
                delta = today_val - prev_val
                pct = (delta / prev_val * 100) if prev_val != 0 else float('inf')
                flag = " ⚠" if abs(pct) > 20 else ""
                print(f"{seg:<18} {col:<30} {prev_val:>8.0f} {today_val:>8.0f} {delta:>+8.0f} {pct:>+7.1f}%{flag}")
                if abs(pct) > 20:
                    anomalies.append((seg, col, pct))

        print()
        if anomalies:
            print(f"WARNING: {len(anomalies)} metric(s) changed >20% day-over-day.")
            print("This may be normal (e.g., deployment start/end dates shifting)")
            print("but review the flagged metrics above.")
        else:
            print("CHECK 2 PASSED: No anomalous day-over-day swings (>20%).")

except Exception as e:
    print(f"Could not fetch previous day data: {e}")
    print("Skipping day-over-day comparison.")

In [ ]:
# --- Table summary + full data for effective_date ---
summary_csv = run_cli(
    f'pharos sql run --sql "SELECT market_segment, COUNT(*) AS total_days, MIN(snapshot_date) AS first_date, MAX(snapshot_date) AS last_date FROM cdt.{TABLE_NAME} WHERE snapshot_date IS NOT NULL GROUP BY market_segment ORDER BY market_segment"',
    fetch_data=True,
)
df_summary = pd.read_csv(StringIO(summary_csv))
print("Table summary:")
print(df_summary.to_string(index=False))

print()

# Full row data for effective_date
today_csv = run_cli(
    f'pharos sql run --sql "SELECT * FROM cdt.{TABLE_NAME} WHERE snapshot_date = DATE \'{effective_date}\'"',
    fetch_data=True,
)
df_full = pd.read_csv(StringIO(today_csv))
print(f"Full data for {effective_date}: {len(df_full)} rows")
df_full